In [2]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

TensorFlow: 2.20.0
NumPy: 2.0.2
Pandas: 2.3.3


In [ ]:
UTKFACE_DIR = "nationality_detection/dataset/UTKFace"
print("Dataset exists:", os.path.exists(UTKFACE_DIR))

utk_files = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    utk_files.extend(glob.glob(os.path.join(UTKFACE_DIR, "**", ext), recursive=True))

print("Total files:", len(utk_files))

Dataset exists: True
Total files: 23708


In [4]:
data = []
for file in utk_files:
    name = os.path.basename(file)
    parts = name.split("_")
    if len(parts) >= 4:
        try:
            age = int(parts[0])
            gender = int(parts[1])
            race = int(parts[2])
            if 0 <= age <= 116 and race in [0, 1, 2, 3, 4]:
                data.append([file, age, gender, race])
        except:
            pass

df = pd.DataFrame(data, columns=["filepath", "age", "gender", "race"])
print("Valid images:", len(df))
print()
print("Race distribution:")
print(df["race"].value_counts().sort_index())

Valid images: 23705

Race distribution:
race
0    10078
1     4526
2     3434
3     3975
4     1692
Name: count, dtype: int64


In [5]:
task_map = {0: 0, 1: 1, 2: 2, 3: 3, 4: 2}
nationality_names = {0: "United States", 1: "African", 2: "Other", 3: "Indian"}

df["nationality_label"] = df["race"].map(task_map)
print(df["nationality_label"].map(nationality_names).value_counts())

nationality_label
United States    10078
Other             5126
African           4526
Indian            3975
Name: count, dtype: int64


In [6]:
nationality_train, nationality_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["nationality_label"]
)
nationality_train = nationality_train.reset_index(drop=True)
nationality_test = nationality_test.reset_index(drop=True)

print("Train:", len(nationality_train))
print("Test:", len(nationality_test))
print()
print("Train distribution:")
print(nationality_train["nationality_label"].map(nationality_names).value_counts())
print()
print("Test distribution:")
print(nationality_test["nationality_label"].map(nationality_names).value_counts())

Train: 18964
Test: 4741

Train distribution:
nationality_label
United States    8062
Other            4101
African          3621
Indian           3180
Name: count, dtype: int64

Test distribution:
nationality_label
United States    2016
Other            1025
African           905
Indian            795
Name: count, dtype: int64


In [7]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=nationality_train["nationality_label"])
nationality_class_weights = dict(zip(classes, weights))
print(nationality_class_weights)

{np.int64(0): np.float64(0.5880674770528405), np.int64(1): np.float64(1.3093068213200774), np.int64(2): np.float64(1.156059497683492), np.int64(3): np.float64(1.4908805031446541)}


In [8]:
IMG_SIZE = 160
BATCH_SIZE = 32

def load_nationality_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image

def create_nationality_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["nationality_label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def process(path, label):
        image = load_nationality_image(path)
        return image, label

    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

nationality_train_ds = create_nationality_dataset(nationality_train, shuffle=True)
nationality_test_ds = create_nationality_dataset(nationality_test)
print(nationality_train_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


I0000 00:00:1789134367.253283      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789134367.255984      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [9]:
nationality_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.RandomContrast(0.10)
])

def augment_nationality(images, labels):
    images = nationality_augmentation(images, training=True)
    return images, labels

nationality_train_aug = nationality_train_ds.map(
    augment_nationality, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

In [10]:
inputs = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)

x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(512, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

nationality_output = tf.keras.layers.Dense(4, activation="softmax", name="nationality")(x)
nationality_model = tf.keras.Model(inputs=inputs, outputs=nationality_output)

nationality_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
nationality_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 80, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 40, 40, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 40, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 20, 20, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 20, 20, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 10, 10, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 10, 10, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 10, 10, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nationality (Dense)             │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,704,900 (6.50 MB)

 Trainable params: 1,702,916 (6.50 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [11]:
nationality_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, mode="max", restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "nationality_detector_final.keras", monitor="val_accuracy", mode="max", save_best_only=True, verbose=1
    )
]

In [12]:
nationality_history = nationality_model.fit(
    nationality_train_aug,
    validation_data=nationality_test_ds,
    epochs=50,
    class_weight=nationality_class_weights,
    callbacks=nationality_callbacks
)

Epoch 1/50
  2/593 ━━━━━━━━━━━━━━━━━━━━ 33s 57ms/step - accuracy: 0.3828 - loss: 1.4215   

I0000 00:00:1789134386.063661     178 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.4520 - loss: 1.2425
Epoch 1: val_accuracy improved from None to 0.55052, saving model to nationality_detector_final.keras

Epoch 1: finished saving model to nationality_detector_final.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 112s 161ms/step - accuracy: 0.5031 - loss: 1.1556 - val_accuracy: 0.5505 - val_loss: 1.0583
Epoch 2/50
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.6035 - loss: 0.9629
Epoch 2: val_accuracy improved from 0.55052 to 0.59101, saving model to nationality_detector_final.keras

Epoch 2: finished saving model to nationality_detector_final.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 64s 106ms/step - accuracy: 0.6281 - loss: 0.9204 - val_accuracy: 0.5910 - val_loss: 1.0591
Epoch 3/50
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.6746 - loss: 0.8286
Epoch 3: val_accuracy improved from 0.59101 to 0.72938, saving model to nationality_detector_final.keras

Epoch 3: finished saving model to nationality_detector_fi

In [13]:
y_true_nationality = []
y_pred_nationality = []

for images, labels in nationality_test_ds:
    predictions = nationality_model.predict(images, verbose=0)
    y_true_nationality.extend(labels.numpy())
    y_pred_nationality.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true_nationality, y_pred_nationality,
    target_names=["United States", "African", "Other", "Indian"]
))
print("Confusion Matrix:")
print(confusion_matrix(y_true_nationality, y_pred_nationality))

               precision    recall  f1-score   support

United States       0.90      0.84      0.87      2016
      African       0.78      0.91      0.84       905
        Other       0.78      0.72      0.75      1025
       Indian       0.73      0.80      0.76       795

     accuracy                           0.82      4741
    macro avg       0.80      0.82      0.81      4741
 weighted avg       0.82      0.82      0.82      4741

Confusion Matrix:
[[1684   85  141  106]
 [  24  821   19   41]
 [ 120   70  741   94]
 [  40   74   43  638]]


In [ ]:
external_images = {
    "American": "nationality_detection/dataset/american.jpg",
    "Asian": "nationality_detection/dataset/asian.jpeg",
    "Indian": "nationality_detection/dataset/indian.jpeg",
    "African": "nationality_detection/dataset/african.jpeg"
}

def predict_nationality_image(image_path):
    image = tf.keras.utils.load_img(image_path, target_size=(160, 160))
    image = tf.keras.utils.img_to_array(image)
    image = np.expand_dims(image, axis=0)
    prediction = nationality_model.predict(image, verbose=0)[0]
    class_id = int(np.argmax(prediction))
    return nationality_names[class_id], float(prediction[class_id]), prediction

for name, path in external_images.items():
    result, confidence, probabilities = predict_nationality_image(path)
    print(name, "->", result, f"{confidence * 100:.2f}%")
    print(probabilities)

American -> United States 97.56%
[9.7556710e-01 1.2585112e-03 2.2874275e-02 3.0012746e-04]
Asian -> United States 38.04%
[0.38035017 0.28158587 0.2920603  0.04600369]
Indian -> African 83.21%
[0.06950054 0.8321023  0.03831118 0.06008593]
African -> Other 51.91%
[0.25584018 0.13873413 0.5190905  0.08633517]


In [15]:
age_df = df[["filepath", "age"]].copy()
age_df = age_df[(age_df["age"] >= 0) & (age_df["age"] <= 116)].reset_index(drop=True)
print("Age samples:", len(age_df))
print(age_df["age"].describe())

Age samples: 23705
count    23705.000000
mean        33.300907
std         19.885708
min          1.000000
25%         23.000000
50%         29.000000
75%         45.000000
max        116.000000
Name: age, dtype: float64


In [16]:
age_train, age_test = train_test_split(age_df, test_size=0.2, random_state=42)
age_train = age_train.reset_index(drop=True)
age_test = age_test.reset_index(drop=True)
print("Age train:", len(age_train))
print("Age test:", len(age_test))

Age train: 18964
Age test: 4741


In [17]:
AGE_IMG_SIZE = 160
AGE_BATCH_SIZE = 32

def load_age_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [AGE_IMG_SIZE, AGE_IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image

def create_age_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["age"].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def process(path, label):
        image = load_age_image(path)
        return image, label

    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    ds = ds.batch(AGE_BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

age_train_ds = create_age_dataset(age_train, shuffle=True)
age_test_ds = create_age_dataset(age_test)
print(age_train_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.float32, name=None))>


In [18]:
age_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.08)
])

def augment_age(images, labels):
    images = age_augmentation(images, training=True)
    return images, labels

age_train_aug = age_train_ds.map(augment_age, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [19]:
age_inputs = tf.keras.Input(shape=(160, 160, 3))
x = tf.keras.layers.Rescaling(1.0 / 255)(age_inputs)

x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(512, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.4)(x)

age_output = tf.keras.layers.Dense(1, activation="linear", name="age")(x)
age_model = tf.keras.Model(age_inputs, age_output)

age_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="mae",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)
age_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 160, 160, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 160, 160, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 80, 80, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 80, 80, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 80, 80, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 40, 40, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 40, 40, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 40, 40, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 20, 20, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 20, 20, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 10, 10, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 10, 10, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 10, 10, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ age (Dense)                     │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,704,129 (6.50 MB)

 Trainable params: 1,702,145 (6.49 MB)

 Non-trainable params: 1,984 (7.75 KB)

In [20]:
age_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=5, mode="min", restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "age_detector_final.keras", monitor="val_mae", mode="min", save_best_only=True, verbose=1
    )
]

In [ ]:
age_history = age_model.fit(
    age_train_aug, validation_data=age_test_ds, epochs=50, callbacks=age_callbacks
)

Epoch 1/50
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 20.3747 - mae: 20.3747
Epoch 1: val_mae improved from None to 14.35421, saving model to age_detector_final.keras

Epoch 1: finished saving model to age_detector_final.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 62s 90ms/step - loss: 16.6230 - mae: 16.6230 - val_loss: 14.3542 - val_mae: 14.3542
Epoch 2/50
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 13.2370 - mae: 13.2370
Epoch 2: val_mae did not improve from 14.35421
593/593 ━━━━━━━━━━━━━━━━━━━━ 50s 82ms/step - loss: 12.7697 - mae: 12.7697 - val_loss: 19.0564 - val_mae: 19.0564
Epoch 3/50
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 11.4269 - mae: 11.4269
Epoch 3: val_mae improved from 14.35421 to 14.05491, saving model to age_detector_final.keras

Epoch 3: finished saving model to age_detector_final.keras
593/593 ━━━━━━━━━━━━━━━━━━━━ 50s 82ms/step - loss: 11.1272 - mae: 11.1272 - val_loss: 14.0549 - val_mae: 14.0549
Epoch 4/50
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - los

In [22]:
age_results = age_model.evaluate(age_test_ds, verbose=1)
print("Age Test Loss:", age_results[0])
print("Age MAE:", age_results[1])

149/149 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 6.1994 - mae: 6.1994
Age Test Loss: 6.199409484863281
Age MAE: 6.199409484863281


In [23]:
def predict_age_image(image_path):
    image = tf.keras.utils.load_img(image_path, target_size=(160, 160))
    image = tf.keras.utils.img_to_array(image)
    image = np.expand_dims(image, axis=0)
    prediction = age_model.predict(image, verbose=0)
    age = float(prediction[0][0])
    age = max(0, min(age, 116))
    return round(age)

for name, path in external_images.items():
    print(name, "-> predicted age:", predict_age_image(path))

American -> predicted age: 38
Asian -> predicted age: 30
Indian -> predicted age: 36
African -> predicted age: 72


In [ ]:
APPAREL_DIR = "nationality_detection/dataset/apparel-dataset"
print("Dataset exists:", os.path.exists(APPAREL_DIR))

dress_files = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    dress_files.extend(glob.glob(os.path.join(APPAREL_DIR, "**", ext), recursive=True))

print("Images found:", len(dress_files))

Images found: 16170


In [ ]:
APPAREL_DIR = "nationality_detection/dataset/apparel-dataset"
dress_data = []

for root, dirs, files in os.walk(APPAREL_DIR):
    folder = os.path.basename(root)
    if "_" not in folder:
        continue
    color = folder.split("_")[0]
    valid_colors = ["black", "blue", "brown", "green", "pink", "red", "silver", "white", "yellow"]
    if color not in valid_colors:
        continue
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            dress_data.append([os.path.join(root, file), color])

dress_df = pd.DataFrame(dress_data, columns=["filepath", "color"])
print("Total images:", len(dress_df))
print()
print(dress_df["color"].value_counts())

Total images: 16170

color
black     3449
blue      2863
red       2399
white     2166
green     1290
yellow    1170
pink      1106
brown      963
silver     764
Name: count, dtype: int64


In [28]:
dress_labels = {"black":0,"blue":1,"brown":2,"green":3,"pink":4,"red":5,"silver":6,"white":7,"yellow":8}
dress_names = {v: k for k, v in dress_labels.items()}
dress_df["color_label"] = dress_df["color"].map(dress_labels)
print(dress_df["color_label"].value_counts().sort_index())

color_label
0    3449
1    2863
2     963
3    1290
4    1106
5    2399
6     764
7    2166
8    1170
Name: count, dtype: int64


In [29]:
dress_train, dress_test = train_test_split(
    dress_df, test_size=0.2, random_state=42, stratify=dress_df["color_label"]
)
dress_train = dress_train.reset_index(drop=True)
dress_test = dress_test.reset_index(drop=True)
print("Train:", len(dress_train))
print("Test:", len(dress_test))

Train: 12936
Test: 3234


In [30]:
DRESS_IMG_SIZE = 128
DRESS_BATCH_SIZE = 32

def load_dress_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [DRESS_IMG_SIZE, DRESS_IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image

def create_dress_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["color_label"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def process(path, label):
        image = load_dress_image(path)
        return image, label

    ds = ds.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2000)
    ds = ds.batch(DRESS_BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

dress_train_ds = create_dress_dataset(dress_train, shuffle=True)
dress_test_ds = create_dress_dataset(dress_test)
print(dress_train_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [31]:
dress_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.10)
])

def augment_dress(images, labels):
    images = dress_augmentation(images, training=True)
    return images, labels

dress_train_aug = dress_train_ds.map(augment_dress, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [32]:
dress_inputs = tf.keras.Input(shape=(128, 128, 3))
x = tf.keras.layers.Rescaling(1.0 / 255)(dress_inputs)

x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D()(x)

x = tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.4)(x)

dress_output = tf.keras.layers.Dense(9, activation="softmax", name="dress_color")(x)
dress_model = tf.keras.Model(dress_inputs, dress_output)

dress_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
dress_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dress_color (Dense)             │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 424,393 (1.62 MB)

 Trainable params: 423,433 (1.62 MB)

 Non-trainable params: 960 (3.75 KB)

In [33]:
dress_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, mode="max", restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "dress_color_detector_final.keras", monitor="val_accuracy", mode="max", save_best_only=True, verbose=1
    )
]

In [34]:
dress_history = dress_model.fit(
    dress_train_aug, validation_data=dress_test_ds, epochs=50, callbacks=dress_callbacks
)

Epoch 1/50
404/405 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.6589 - loss: 1.1135

2026-09-11 14:37:40.002297: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-11 14:37:40.145490: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


405/405 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.6591 - loss: 1.1128
Epoch 1: val_accuracy improved from None to 0.32653, saving model to dress_color_detector_final.keras

Epoch 1: finished saving model to dress_color_detector_final.keras
405/405 ━━━━━━━━━━━━━━━━━━━━ 46s 90ms/step - accuracy: 0.7460 - loss: 0.8496 - val_accuracy: 0.3265 - val_loss: 1.8636
Epoch 2/50
404/405 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8264 - loss: 0.5796
Epoch 2: val_accuracy improved from 0.32653 to 0.85931, saving model to dress_color_detector_final.keras

Epoch 2: finished saving model to dress_color_detector_final.keras
405/405 ━━━━━━━━━━━━━━━━━━━━ 25s 58ms/step - accuracy: 0.8319 - loss: 0.5484 - val_accuracy: 0.8593 - val_loss: 0.4463
Epoch 3/50
403/405 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.8448 - loss: 0.5014
Epoch 3: val_accuracy improved from 0.85931 to 0.87322, saving model to dress_color_detector_final.keras

Epoch 3: finished saving model to dress_color_detector_final.ke

In [35]:
y_true_dress = []
y_pred_dress = []

for images, labels in dress_test_ds:
    predictions = dress_model.predict(images, verbose=0)
    y_true_dress.extend(labels.numpy())
    y_pred_dress.extend(np.argmax(predictions, axis=1))

print(classification_report(
    y_true_dress, y_pred_dress,
    target_names=["black","blue","brown","green","pink","red","silver","white","yellow"]
))
print("Confusion Matrix:")
print(confusion_matrix(y_true_dress, y_pred_dress))

              precision    recall  f1-score   support

       black       0.97      0.96      0.96       690
        blue       0.96      0.97      0.97       572
       brown       0.97      0.93      0.95       193
       green       0.90      0.93      0.91       258
        pink       0.93      0.95      0.94       221
         red       0.98      0.98      0.98       480
      silver       0.77      0.91      0.83       153
       white       0.98      0.89      0.93       433
      yellow       0.98      0.97      0.97       234

    accuracy                           0.95      3234
   macro avg       0.94      0.94      0.94      3234
weighted avg       0.95      0.95      0.95      3234

Confusion Matrix:
[[664   4   2  13   1   2   3   1   0]
 [  6 557   0   5   0   1   0   3   0]
 [  4   2 180   2   0   2   1   0   2]
 [  5   3   1 240   1   0   5   2   1]
 [  0   0   0   1 211   1   7   1   0]
 [  1   0   1   0   7 471   0   0   0]
 [  3   3   0   3   3   0 139   1   1]
 [  

In [36]:
def predict_dress_image(image_path):
    image = tf.keras.utils.load_img(image_path, target_size=(128, 128))
    image = tf.keras.utils.img_to_array(image)
    image = np.expand_dims(image, axis=0)
    prediction = dress_model.predict(image, verbose=0)[0]
    class_id = int(np.argmax(prediction))
    return dress_names[class_id], float(prediction[class_id]), prediction

In [37]:
nationality_model.save("nationality_detector_final.keras")  
age_model.save("age_detector_final.keras")
dress_model.save("dress_color_detector_final.keras")

print("Nationality model saved")
print("Age model saved")
print("Dress colour model saved")

Nationality model saved
Age model saved
Dress colour model saved
